# Install KenLM Library

In [10]:
!apt-get update
!apt-get install -y libboost-program-options-dev libboost-system-dev

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,388 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,67

In [11]:
!wget -O - https://kheafield.com/code/kenlm.tar.gz |tar xz
!mv kenlm kenlm_tool
!mkdir kenlm_tool/build

--2026-02-06 07:20:11--  https://kheafield.com/code/kenlm.tar.gz
Resolving kheafield.com (kheafield.com)... 129.80.89.152, 2603:c020:4009:8710:ca:11:17:0
Connecting to kheafield.com (kheafield.com)|129.80.89.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 491888 (480K) [application/octet-stream]
Saving to: ‘STDOUT’

-                   100%[===================>] 480.36K  --.-KB/s    in 0.05s   

2026-02-06 07:20:11 (8.81 MB/s) - written to stdout [491888/491888]



In [12]:
%cd kenlm_tool/build
!cmake ..
!make -j2

/content/kenlm_tool/build/kenlm_tool/build
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Could NOT find Eigen3 (missing: Eigen3_DIR)
CMake Warning (dev) at CMakeLists.t

# Download Data

In [13]:
!gdown 1g6zRU45wZqZjLyOPt_4BNft8DXUuxGgQ

Downloading...
From: https://drive.google.com/uc?id=1g6zRU45wZqZjLyOPt_4BNft8DXUuxGgQ
To: /content/kenlm_tool/build/kenlm_tool/build/gigaword3_nyt_eng_200001.tar.gz
100% 17.6M/17.6M [00:00<00:00, 246MB/s]


In [14]:
!tar -xf gigaword3_nyt_eng_200001.tar.gz

In [15]:
ls

CMakeCache.txt  CMakeFiles/  gigaword3_nyt_eng_200001.tar.gz  nyt_eng_200001


# Install KenLM in Python

In [16]:
!pip install https://github.com/kpu/kenlm/archive/master.zip


  Using cached https://github.com/kpu/kenlm/archive/master.zip (553 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Train LM with KenLM

In [17]:
!bin/lmplz -o 5 --text nyt_eng_200001 --arpa fivegram.arpa

/bin/bash: line 1: bin/lmplz: No such file or directory


In [18]:
ls

CMakeCache.txt  CMakeFiles/  gigaword3_nyt_eng_200001.tar.gz  nyt_eng_200001


In [20]:
import kenlm
model = kenlm.Model('fivegram.arpa')

OSError: Cannot read model 'fivegram.arpa' (util/file.cc:76 in int util::OpenReadOrThrow(const char*) threw ErrnoException because `-1 == (ret = open(name, 00))'. No such file or directory while opening /content/kenlm_tool/build/kenlm_tool/build/fivegram.arpa)

In [ ]:
import math
import pandas as pd
def print_score(model, s):
  tokens = s.split(' ')
  log_score = 0.0
  rows = []
  for i, (logprob, length, oov) in enumerate(model.full_scores(s)):
    if i < len(tokens):
      row = {'token': tokens[i], 'probability': math.exp(logprob), 'Is OOV?': oov}
    else:
      row = {'token': 'END', 'probability': math.exp(logprob), 'Is OOV?': oov}
    rows.append(row)
    log_score += logprob
  print ('Log probability = ', log_score)
  return pd.DataFrame(rows)

In [ ]:
print_score(model, 'I look forward to meeting you')

In [ ]:
print_score(model, 'I look forward to meet you')

In [ ]:
print_score(model, 'The president ceases trading to China yesterday')

In [ ]:
print_score(model, 'The president ceased trading with China yesterday')

In [ ]:
print_score(model, 'This essay will discuss about the current issues of climate change')

In [ ]:
print_score(model, 'This essay will discuss the current issues of climate change')

In [ ]:
print_score(model, "My favorite things are eating my family and not using punctuation")

In [ ]:
print_score(model, "Colorless green ideas sleep furiously")